# 🚛 Analiz 2: Şoför Etkisi — Rotasyon, Sürüş Stili, ML Feature

**Hedef:** Şoför pattern'lerinin arıza ciddiyetine etkisini ölçmek ve ML modeline güçlü feature üretmek.

**Metodoloji notu:**
- İETT araçları günde 2-3 vardiya, 2-3 farklı şoför kullanır
- SOFOR_SICILNO arıza anındaki son aktif şoför
- arac_gunluk_hatlar.csv sefer verisi attribution belirsizliğini azaltır
- ML kararı VERİYE göre verilir (şartlama yok)

---

## 1. Veri Yükleme + Şoför Profil İstatistikleri

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# Veri yukleme
df = pd.read_csv('../panel_data/temiz_veri/ariza_model.csv')
df['OLAYTARIHI'] = pd.to_datetime(df['OLAYTARIHI'], format='mixed')

# Sefer verisi: arac gunluk hat kullanim
arac_hatlar = pd.read_csv('../panel_data/temiz_veri/arac_gunluk_hatlar.csv')
SEFER_KOL = [c for c in arac_hatlar.columns if 'SEFER' in c.upper()][0]
arac_hatlar['TARIH'] = pd.to_datetime(arac_hatlar['TARIH'], format='mixed')

print('=== TEMEL ISTATISTIK ===')
print(f'Toplam ariza:    {len(df):,}')
print(f'Toplam sefer kaydi: {len(arac_hatlar):,}')
print(f'Benzersiz arac:  {df["KAPINO"].nunique():,}')
print(f'Benzersiz sofor: {df["SOFOR_SICILNO"].nunique():,}')
print(f'Sofor/Arac orani: {df["SOFOR_SICILNO"].nunique() / df["KAPINO"].nunique():.2f}')
print(f'Veri penceresi: {(df["OLAYTARIHI"].max() - df["OLAYTARIHI"].min()).days} gun')
print()

# Arac basina toplam sefer (180 gun)
arac_sefer = arac_hatlar.groupby('KAPINO')[SEFER_KOL].sum().reset_index()
arac_sefer.columns = ['KAPINO', 'toplam_sefer']
print('=== ARAC BASINA TOPLAM SEFER (180g) ===')
print(arac_sefer['toplam_sefer'].describe([0.1, 0.5, 0.9]).round(0).to_string())

# Sofor profili (ariza datasi)
sofor_prof = df.groupby('SOFOR_SICILNO').agg(
    toplam_ariza=('KAPINO', 'count'),
    farkli_arac=('KAPINO', 'nunique'),
    ort_skor=('ciddiyet_skoru', 'mean'),
    ciddi_oran=('ciddi_ariza', 'mean'),
).reset_index()

# Sofor basina sefer agirligi: bu soforun cesnistirdigi araclarin toplam seferi (paylaşımlı)
# Yani sofor X arac Y'ye bindigi gun, Y'nin o gunki seferi paylasiliyor
# Basitlestirme: sofor sicilno bazinda toplam ariza zaten kullanım proxy'si
# Daha guzel: sofor basina toplam_sefer = farkli_arac × ortalama sefer
sofor_arac_eslesme = df.groupby(['SOFOR_SICILNO', 'KAPINO']).size().reset_index(name='eslesme_n')
sofor_arac_eslesme = sofor_arac_eslesme.merge(arac_sefer, on='KAPINO', how='left')
# Sofor basina toplam ariza ile arac kullanim hacmi orantili
sofor_kullanim = sofor_arac_eslesme.groupby('SOFOR_SICILNO').agg(
    sofor_toplam_arac_sefer=('toplam_sefer', 'sum'),
    ana_arac_sefer=('toplam_sefer', 'max'),
).reset_index()
sofor_prof = sofor_prof.merge(sofor_kullanim, on='SOFOR_SICILNO', how='left')

# YEDEK SOFOR tespiti (50+ farkli arac VEYA aşırı düşük arıza/araç orani)
sofor_prof['ariza_per_arac'] = sofor_prof['toplam_ariza'] / sofor_prof['farkli_arac']
yedek = sofor_prof[sofor_prof['farkli_arac'] >= 50]
normal = sofor_prof[sofor_prof['farkli_arac'] < 50]

print()
print('=== SOFOR BASINA FARKLI ARAC SAYISI ===')
print(sofor_prof['farkli_arac'].describe([0.1, 0.5, 0.9, 0.95, 0.99]).round(1).to_string())

print()
print(f'=== YEDEK/HAVUZ SOFOR (>=50 farkli arac) ===')
print(f'Sayisi: {len(yedek):,} ({len(yedek)/len(sofor_prof)*100:.1f}%)')
print(f'Toplam ariza katkisi: {yedek["toplam_ariza"].sum():,} ({yedek["toplam_ariza"].sum()/len(df)*100:.1f}%)')
print(f'Median ariza/arac orani: {yedek["ariza_per_arac"].median():.2f}')
print(f'Normal sofor median ariza/arac: {normal["ariza_per_arac"].median():.2f}')

print()
print('=== METHODOLOJI NOTU ===')
print('IETT araclari gunde 2-3 vardiya, 2-3 farkli sofor kullanir.')
print('SOFOR_SICILNO arıza anindaki son aktif sofor olarak yazilir.')
print('Sefer verisi (arac_gunluk_hatlar) attribution belirsizligini azaltir:')
print('  - Sofor basina ortalama sefer (paylasimli)')
print('  - Yedek/havuz sofor tespiti (>=50 arac)')
print('  - Sefer normalize edilmis ariza orani (sonraki bolumlerde)')


=== TEMEL ISTATISTIK ===
Toplam ariza:    58,559
Toplam sefer kaydi: 1,320,647
Benzersiz arac:  3,509
Benzersiz sofor: 5,910
Sofor/Arac orani: 1.68
Veri penceresi: 180 gun

=== ARAC BASINA TOPLAM SEFER (180g) ===
count    6761.0
mean     1442.0
std       413.0
min         1.0
10%       954.0
50%      1451.0
90%      1871.0
max      3720.0

=== SOFOR BASINA FARKLI ARAC SAYISI ===
count    5910.0
mean        8.4
std         7.7
min         1.0
10%         2.0
50%         7.0
90%        17.0
95%        20.0
99%        29.0
max       350.0

=== YEDEK/HAVUZ SOFOR (>=50 farkli arac) ===
Sayisi: 3 (0.1%)
Toplam ariza katkisi: 606 (1.0%)
Median ariza/arac orani: 1.04
Normal sofor median ariza/arac: 1.06

=== METHODOLOJI NOTU ===
IETT araclari gunde 2-3 vardiya, 2-3 farkli sofor kullanir.
SOFOR_SICILNO arıza anindaki son aktif sofor olarak yazilir.
Sefer verisi (arac_gunluk_hatlar) attribution belirsizligini azaltir:
  - Sofor basina ortalama sefer (paylasimli)
  - Yedek/havuz sofor tespiti (>=

---

## 2. Şoför Yoğunlaşma Metriği (Sabit vs Göçebe)
Farklı araç sayısı yerine **ana_arac_orani** (ana araç kullanım yüzdesi) ile yoğunlaşma ölç.

In [2]:
# BOLUM 2: Sofor Yogunlasma Metrigi (Sabit vs Gocebe)
# Sorun: 180 gun icinde 8.4 farkli arac kullanmak vardiya rotasyonu icin normal mi anormal mi?
# Daha iyi metrik: ana_arac_orani — sofor en cok kullandigi araca toplam kullanımının % kacini ayirdi?

# Her sofor X arac ciftinin kullanim sayisi
sa_eslesme = df.groupby(['SOFOR_SICILNO', 'KAPINO']).size().reset_index(name='kullanim_n')

# Her sofor icin: en cok kullanilan arac % kac kullaniminin
sofor_max = sa_eslesme.groupby('SOFOR_SICILNO').agg(
    max_arac_kullanim=('kullanim_n', 'max'),
    toplam_kullanim=('kullanim_n', 'sum'),
).reset_index()
sofor_max['ana_arac_orani'] = sofor_max['max_arac_kullanim'] / sofor_max['toplam_kullanim']

# Yogunlasma sinifi
sofor_max['yogunlasma_sinif'] = pd.cut(sofor_max['ana_arac_orani'],
    bins=[-0.01, 0.20, 0.40, 0.60, 1.01],
    labels=['Çok Göçebe (<20%)', 'Göçebe (20-40%)', 'Orta (40-60%)', 'Sabit (60%+)'])

print('=== SOFOR YOGUNLASMA DAGILIMI ===')
print(sofor_max['ana_arac_orani'].describe([0.1, 0.25, 0.5, 0.75, 0.9]).round(3).to_string())
print()
print('=== YOGUNLASMA SINIF SAYISI ===')
print(sofor_max['yogunlasma_sinif'].value_counts().to_string())

# Ciddiyet × yogunlasma karsilastirmasi
sofor_max = sofor_max.merge(sofor_prof[['SOFOR_SICILNO','ort_skor','ciddi_oran','toplam_ariza','farkli_arac']],
                              on='SOFOR_SICILNO', how='left')
# Min 5 ariza filtre
sofor_max_filt = sofor_max[sofor_max['toplam_ariza'] >= 5]
print()
print(f'Analiz seti: {len(sofor_max_filt):,} sofor (min 5 ariza)')

# Yogunlasma × ciddiyet
print()
print('=== YOGUNLASMA SINIF × ORT_SKOR ===')
ozet = sofor_max_filt.groupby('yogunlasma_sinif', observed=False).agg(
    n=('SOFOR_SICILNO', 'count'),
    ort_skor=('ort_skor', 'mean'),
    ort_ciddi=('ciddi_oran', 'mean'),
    ort_farkli_arac=('farkli_arac', 'mean'),
).round(3)
print(ozet.to_string())

# Anova
gruplar = [g['ort_skor'].dropna().values for _, g in sofor_max_filt.groupby('yogunlasma_sinif', observed=False)
            if g['ort_skor'].dropna().shape[0] >= 10]
if len(gruplar) >= 2:
    f, p = stats.f_oneway(*gruplar)
    print(f'\nANOVA F={f:.2f}, p={p:.6f}')
    if p < 0.05:
        print('VERIYE GORE: Yogunlasma siniflari arasinda ciddiyet ANLAMLI farkli.')
    else:
        print('VERIYE GORE: Yogunlasma siniflari arasinda anlamli fark YOK.')

# Korelasyon: ana_arac_orani × ort_skor
sub = sofor_max_filt.dropna(subset=['ana_arac_orani', 'ort_skor'])
r, p = stats.pearsonr(sub['ana_arac_orani'], sub['ort_skor'])
rs, ps = stats.spearmanr(sub['ana_arac_orani'], sub['ort_skor'])
print()
print(f'=== KORELASYON: ana_arac_orani × ort_skor ===')
print(f'Pearson  r={r:+.3f}, p={p:.4f}')
print(f'Spearman rs={rs:+.3f}, p={ps:.4f}')


=== SOFOR YOGUNLASMA DAGILIMI ===
count    5910.000
mean        0.320
std         0.267
min         0.010
10%         0.077
25%         0.118
50%         0.200
75%         0.500
90%         0.750
max         1.000

=== YOGUNLASMA SINIF SAYISI ===
yogunlasma_sinif
Çok Göçebe (<20%)    2999
Göçebe (20-40%)      1237
Sabit (60%+)          896
Orta (40-60%)         778

Analiz seti: 4,615 sofor (min 5 ariza)

=== YOGUNLASMA SINIF × ORT_SKOR ===
                      n  ort_skor  ort_ciddi  ort_farkli_arac
yogunlasma_sinif                                             
Çok Göçebe (<20%)  2999     3.636      0.393           12.533
Göçebe (20-40%)     770     3.597      0.363            7.240
Orta (40-60%)       440     3.566      0.353            5.482
Sabit (60%+)        406     3.543      0.327            2.968

ANOVA F=2.89, p=0.034345
VERIYE GORE: Yogunlasma siniflari arasinda ciddiyet ANLAMLI farkli.

=== KORELASYON: ana_arac_orani × ort_skor ===
Pearson  r=-0.042, p=0.0041
Spearman rs=-0

---

## 3. Araç Başına Şoför Rotasyonu
Bir araçta kaç farklı şoför çalışmış? Sefer × Şoför confounder kontrolü.

In [3]:
# BOLUM 3: Arac Basina Kac Sofor Calismis (Rotasyon Yogunlugu)
# Bir aracta kac farkli sofor calismis? Bu rotasyon yogunlugu metriği.
# Sefer × Sofor ratio confounder kontrolu icin onemli.

arac_sofor = df.groupby('KAPINO').agg(
    sofor_n=('SOFOR_SICILNO', 'nunique'),
    toplam_ariza=('SOFOR_SICILNO', 'count'),
    ort_skor=('ciddiyet_skoru', 'mean'),
    ciddi_oran=('ciddi_ariza', 'mean'),
).reset_index()

# Arac toplam sefer (paylasilan kullanım)
arac_sofor = arac_sofor.merge(arac_sefer, on='KAPINO', how='left')
arac_sofor['arac_sefer_per_sofor'] = arac_sofor['toplam_sefer'] / arac_sofor['sofor_n']

print('=== ARAC BASINA SOFOR SAYISI (180g) ===')
print(arac_sofor['sofor_n'].describe([0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).round(1).to_string())
print()
print('=== ARAC BASINA TOPLAM SEFER ===')
print(arac_sofor['toplam_sefer'].describe([0.1, 0.5, 0.9]).round(0).to_string())

# Rotasyon yogunlugu siniflari
arac_sofor['rotasyon_sinif'] = pd.cut(arac_sofor['sofor_n'],
    bins=[-0.01, 5, 15, 30, 1000],
    labels=['Düşük (≤5)', 'Orta (6-15)', 'Yüksek (16-30)', 'Aşırı (30+)'])

print()
print('=== ROTASYON SINIF × CIDDIYET ===')
ozet = arac_sofor.groupby('rotasyon_sinif', observed=False).agg(
    n_arac=('KAPINO', 'count'),
    ort_sofor_n=('sofor_n', 'mean'),
    ort_skor=('ort_skor', 'mean'),
    ort_ciddi=('ciddi_oran', 'mean'),
    ort_sefer=('toplam_sefer', 'mean'),
).round(2)
print(ozet.to_string())

# ANOVA
gruplar = [g['ort_skor'].dropna().values for _, g in arac_sofor.groupby('rotasyon_sinif', observed=False)
            if g['ort_skor'].dropna().shape[0] >= 5]
if len(gruplar) >= 2:
    f, p = stats.f_oneway(*gruplar)
    print(f'\nANOVA F={f:.2f}, p={p:.6f}')

# Korelasyon: arac sofor_n × ort_skor + sefer kontrolu
sub = arac_sofor.dropna(subset=['sofor_n', 'ort_skor', 'toplam_sefer'])
r1, p1 = stats.pearsonr(sub['sofor_n'], sub['ort_skor'])
r2, p2 = stats.pearsonr(sub['toplam_sefer'], sub['ort_skor'])
r3, p3 = stats.pearsonr(sub['sofor_n'], sub['toplam_sefer'])

print()
print('=== KORELASYON MATRISI ===')
print(f'sofor_n     × ort_skor:    r={r1:+.3f}  p={p1:.4f}')
print(f'toplam_sefer× ort_skor:    r={r2:+.3f}  p={p2:.4f}')
print(f'sofor_n     × sefer:        r={r3:+.3f}  p={p3:.4f}  (CONFOUNDER!)')

# Partial korelasyon: sefer kontrol altinda sofor_n -> ort_skor
# Basit lineer regresyon ile residualler
from numpy.linalg import lstsq
X = sub[['toplam_sefer']].values
res_sofor = sub['sofor_n'].values - X @ lstsq(X, sub['sofor_n'].values, rcond=None)[0]
res_skor  = sub['ort_skor'].values - X @ lstsq(X, sub['ort_skor'].values, rcond=None)[0]
r_partial = np.corrcoef(res_sofor, res_skor)[0,1]
print()
print(f'Partial korelasyon sofor_n × ort_skor | sefer: r={r_partial:+.3f}')
if abs(r_partial) < 0.05:
    print('  -> Sefer kontrol altinda sofor_n etkisi neredeyse SIFIRA inivor.')
    print('  -> "Cok sofor = ciddi ariza" iliskisi aslinda "cok sefer"\'den geliyor.')


=== ARAC BASINA SOFOR SAYISI (180g) ===
count    3509.0
mean       14.2
std         9.2
min         1.0
10%         4.0
25%         7.0
50%        12.0
75%        20.0
90%        27.0
95%        31.0
max        57.0

=== ARAC BASINA TOPLAM SEFER ===
count    3509.0
mean     1339.0
std       340.0
min         1.0
10%       907.0
50%      1371.0
90%      1724.0
max      3016.0

=== ROTASYON SINIF × CIDDIYET ===
                n_arac  ort_sofor_n  ort_skor  ort_ciddi  ort_sefer
rotasyon_sinif                                                     
Düşük (≤5)         608         3.71      3.35       0.30    1475.22
Orta (6-15)       1559         9.89      3.56       0.36    1349.07
Yüksek (16-30)    1127        21.70      3.65       0.40    1252.34
Aşırı (30+)        215        35.63      3.68       0.41    1335.55

ANOVA F=24.49, p=0.000000

=== KORELASYON MATRISI ===
sofor_n     × ort_skor:    r=+0.118  p=0.0000
toplam_sefer× ort_skor:    r=-0.147  p=0.0000
sofor_n     × sefer:        r=-0

---

## 4. Vardiya Proxy (Saat Verisi)
Vardiya verisi yok, ama arıza saati ile sabah/akşam/gece bandı türetilebilir.

In [4]:
# BOLUM 4: Vardiya Proxy (Saat Verisi)
# Vardiya verisi yok, ama arıza saatinden vardiya proxy üretebiliriz.
# IETT operasyonel: Sabah (06-14), Akşam (14-22), Gece (22-06)

def vardiya_belirle(saat):
    if 6 <= saat < 14: return 'Sabah'
    elif 14 <= saat < 22: return 'Aksam'
    else: return 'Gece'

df['vardiya'] = df['SAAT'].apply(vardiya_belirle)

print('=== VARDIYA × ARIZA DAGILIMI ===')
print(df['vardiya'].value_counts().to_string())
print()

# Vardiya × ciddiyet
print('=== VARDIYA × CIDDIYET ===')
print(df.groupby('vardiya').agg(
    n=('SOFOR_SICILNO', 'count'),
    ort_skor=('ciddiyet_skoru', 'mean'),
    ort_ciddi=('ciddi_ariza', 'mean'),
).round(3).to_string())

# Sofor başına dominant vardiya
sofor_vardiya = df.groupby(['SOFOR_SICILNO', 'vardiya']).size().reset_index(name='n')
sofor_dominant = sofor_vardiya.loc[sofor_vardiya.groupby('SOFOR_SICILNO')['n'].idxmax()]
sofor_dominant = sofor_dominant.rename(columns={'vardiya': 'dominant_vardiya'})[['SOFOR_SICILNO','dominant_vardiya']]

# Sofor vardiya yogunluğu (en cok kullanilan vardiyanin yuzdesi)
sofor_vardiya_total = df.groupby('SOFOR_SICILNO').size().reset_index(name='toplam')
sofor_vardiya_max  = df.groupby(['SOFOR_SICILNO','vardiya']).size().groupby('SOFOR_SICILNO').max().reset_index(name='max_vardiya_n')
sofor_vardiya_yog = sofor_vardiya_total.merge(sofor_vardiya_max, on='SOFOR_SICILNO')
sofor_vardiya_yog['vardiya_yogunluk'] = sofor_vardiya_yog['max_vardiya_n'] / sofor_vardiya_yog['toplam']

print()
print('=== SOFOR VARDIYA YOGUNLUGU (max vardiya % kullanımı) ===')
print(sofor_vardiya_yog['vardiya_yogunluk'].describe([0.1, 0.5, 0.9]).round(3).to_string())

# Sofor dominant vardiya × ciddiyet
sofor_full = sofor_prof.merge(sofor_dominant, on='SOFOR_SICILNO', how='left')
sofor_full = sofor_full.merge(sofor_vardiya_yog[['SOFOR_SICILNO','vardiya_yogunluk']], on='SOFOR_SICILNO', how='left')
sofor_full_f = sofor_full[sofor_full['toplam_ariza'] >= 5]

print()
print('=== SOFOR DOMINANT VARDIYA × CIDDIYET ===')
print(sofor_full_f.groupby('dominant_vardiya').agg(
    n=('SOFOR_SICILNO','count'),
    ort_skor=('ort_skor','mean'),
    ort_ciddi=('ciddi_oran','mean'),
).round(3).to_string())

# Vardiya yogunluk x ciddiyet
sofor_full_f['vy_bant'] = pd.cut(sofor_full_f['vardiya_yogunluk'],
    bins=[-0.01, 0.5, 0.7, 0.9, 1.01],
    labels=['Karisik (<0.5)', 'Yari (0.5-0.7)', 'Ag.Yogun (0.7-0.9)', 'Cok Yog (0.9+)'])
print()
print('=== VARDIYA YOGUNLUK BANT × CIDDIYET ===')
print(sofor_full_f.groupby('vy_bant', observed=False).agg(
    n=('SOFOR_SICILNO','count'),
    ort_skor=('ort_skor','mean'),
).round(3).to_string())


=== VARDIYA × ARIZA DAGILIMI ===
vardiya
Sabah    29009
Aksam    26155
Gece      3395

=== VARDIYA × CIDDIYET ===
             n  ort_skor  ort_ciddi
vardiya                            
Aksam    26155     3.574      0.382
Gece      3395     3.816      0.433
Sabah    29009     3.618      0.373

=== SOFOR VARDIYA YOGUNLUGU (max vardiya % kullanımı) ===
count    5910.000
mean        0.657
std         0.162
min         0.333
10%         0.500
50%         0.619
90%         1.000
max         1.000

=== SOFOR DOMINANT VARDIYA × CIDDIYET ===
                     n  ort_skor  ort_ciddi
dominant_vardiya                           
Aksam             2258     3.608      0.377
Gece                64     3.842      0.465
Sabah             2293     3.615      0.376

=== VARDIYA YOGUNLUK BANT × CIDDIYET ===
                       n  ort_skor
vy_bant                           
Karisik (<0.5)       969     3.674
Yari (0.5-0.7)      2570     3.604
Ag.Yogun (0.7-0.9)   933     3.589
Cok Yog (0.9+)       14

---

## 5. Random Null Model — Şoför Etkisi Gerçek mi?
Şoför varyansı rastgele permutasyondan anlamlı yüksek mi? 100 permutasyon testi.

In [5]:
# BOLUM 5: Sofor Etkisi Random mi Gercek mi (Null Model)
# Soru: Sofor ciddi_oran dagilimi sansa bagli olabilir mi?
# Yontem: Soforleri rastgele permute et, gercek varyansla karsilastir.

sofor_filt = sofor_prof[sofor_prof['toplam_ariza'] >= 10].copy()
gercek_var = sofor_filt['ort_skor'].var()

# Permutasyon: ciddiyet skorlarini sofor sicilno'lara rastgele dagit
np.random.seed(42)
permute_var = []
for _ in range(100):
    df_p = df.copy()
    df_p['SOFOR_SICILNO'] = np.random.permutation(df_p['SOFOR_SICILNO'].values)
    sofor_p = df_p.groupby('SOFOR_SICILNO').agg(
        n=('ciddiyet_skoru','count'),
        ort=('ciddiyet_skoru','mean'),
    ).reset_index().query('n >= 10')
    permute_var.append(sofor_p['ort'].var())

permute_var = np.array(permute_var)
print('=== SOFOR ETKISI NULL TEST (100 PERMUTASYON) ===')
print(f'Gercek sofor ort_skor varyansi: {gercek_var:.4f}')
print(f'Permute median varyans:          {np.median(permute_var):.4f}')
print(f'Permute P95 varyans:             {np.percentile(permute_var, 95):.4f}')
print(f'Gercek varyans permute %X yuksek: {(gercek_var > permute_var).mean()*100:.0f}%')

if gercek_var > np.percentile(permute_var, 95):
    print('VERIYE GORE: Sofor varyansi random permutasyondan ANLAMLI YUKSEK.')
    print('  -> Soforler arasi ciddiyet farkı GERCEK, sansa bagli degil.')
else:
    print('VERIYE GORE: Sofor varyansi random ile uyumlu.')
    print('  -> Sofor etkisi sansa bagli olabilir, ML feature olarak guvensiz.')


=== SOFOR ETKISI NULL TEST (100 PERMUTASYON) ===
Gercek sofor ort_skor varyansi: 0.3676
Permute median varyans:          0.2440
Permute P95 varyans:             0.2534
Gercek varyans permute %X yuksek: 100%
VERIYE GORE: Sofor varyansi random permutasyondan ANLAMLI YUKSEK.
  -> Soforler arasi ciddiyet farkı GERCEK, sansa bagli degil.


---

## 6. Şoför-Araç Uyumu (Süreklilik Bonusu)
Aynı şoför aynı araca çok bindiyse daha az ciddi arıza yapar mı? (Tanıdık araç hipotezi)

In [6]:
# BOLUM 6: Sofor-Arac Uyumu (Süreklilik Bonusu Var Mi?)
# Soru: Bir sofor ayni araca cok bindiyse o aracta daha az ariza yapar mi?
# (Tanidik arac hipotezi)

# Her sofor-arac cifti icin: kullanim sayisi ve ortalama ciddi_oran
sa_full = df.groupby(['SOFOR_SICILNO', 'KAPINO']).agg(
    eslesme_n=('ciddi_ariza', 'count'),
    ort_skor=('ciddiyet_skoru', 'mean'),
    ciddi_oran=('ciddi_ariza', 'mean'),
).reset_index()

# Bu sofor bu araca toplamda kac arıza yapmis?
# eslesme_n = bu sofor + bu arac ciftinde toplam ariza
# Filtre: min 3 eslesme (yetersiz veri eliminate)
sa_filt = sa_full[sa_full['eslesme_n'] >= 3].copy()
print(f'Sofor-Arac cifti (>=3 eslesme): {len(sa_filt):,}')

# Eslesme bant analizi
sa_filt['eslesme_bant'] = pd.cut(sa_filt['eslesme_n'],
    bins=[2, 5, 10, 20, 1000],
    labels=['3-5 kez', '6-10 kez', '11-20 kez', '20+ kez'])
print()
print('=== ESLESME BANT × CIDDIYET ===')
print(sa_filt.groupby('eslesme_bant', observed=False).agg(
    n_cift=('ort_skor','count'),
    ort_skor=('ort_skor','mean'),
    ort_ciddi=('ciddi_oran','mean'),
).round(3).to_string())

# Korelasyon: eslesme_n × ort_skor
r, p = stats.pearsonr(sa_filt['eslesme_n'], sa_filt['ort_skor'])
rs, ps = stats.spearmanr(sa_filt['eslesme_n'], sa_filt['ort_skor'])
print()
print('=== KORELASYON: eslesme_n × ciddiyet ===')
print(f'Pearson  r={r:+.4f}, p={p:.4f}')
print(f'Spearman rs={rs:+.4f}, p={ps:.4f}')

if abs(r) > 0.05 and p < 0.05:
    yon = 'POZITIF' if r > 0 else 'NEGATIF'
    print(f'VERIYE GORE: Eslesme × ciddiyet {yon} ANLAMLI iliski.')
    if r < 0:
        print('  -> Tanidik arac = daha az ciddi ariza (süreklilik bonusu var)')
    else:
        print('  -> Cok kullanim = daha cok ciddi ariza (yorgunluk veya aracin sorunlu olmasi)')
else:
    print(f'VERIYE GORE: Eslesme × ciddiyet anlamsiz (p={p:.4f}).')


Sofor-Arac cifti (>=3 eslesme): 1,550

=== ESLESME BANT × CIDDIYET ===
              n_cift  ort_skor  ort_ciddi
eslesme_bant                             
3-5 kez         1158     3.509      0.325
6-10 kez         354     3.509      0.333
11-20 kez         38     3.446      0.327
20+ kez            0       NaN        NaN

=== KORELASYON: eslesme_n × ciddiyet ===
Pearson  r=-0.0140, p=0.5826
Spearman rs=+0.0209, p=0.4108
VERIYE GORE: Eslesme × ciddiyet anlamsiz (p=0.5826).


---

## 7. Şoför × Sistem Detayı
Hangi sistem arızaları şoföre bağlı, hangileri araca bağlı? Varyans karşılaştırma.

In [7]:
# BOLUM 7: Sofor X Sistem Detayi
# Soru: Hangi sistem arızalari sofor pattern'ina bagli? Hangi sistemler arac'a bagli?
# Yontem: Sofor varyansi vs Arac varyansi (her sistem icin)

print('=== SISTEM BAZINDA SOFOR/ARAC VARYANS ANALIZI ===')
print('Her sistem icin: sofor bazinda vs arac bazinda ciddi_oran varyansı')
print()
print(f'{"Sistem":35s}  ort_ciddi  sof_var  arac_var  oran(sof/arac)')

sistem_sonuc = []
for kat in df['ARIZAUSTKODTANIM'].value_counts().head(15).index:
    sub = df[df['ARIZAUSTKODTANIM'] == kat]
    if len(sub) < 200: continue

    # Sofor bazinda dagilim
    sofor_kat = sub.groupby('SOFOR_SICILNO').agg(
        n=('ciddi_ariza','count'),
        ciddi=('ciddi_ariza','mean'),
    ).query('n >= 3')

    # Arac bazinda dagilim
    arac_kat = sub.groupby('KAPINO').agg(
        n=('ciddi_ariza','count'),
        ciddi=('ciddi_ariza','mean'),
    ).query('n >= 3')

    if len(sofor_kat) < 10 or len(arac_kat) < 10: continue

    sof_var = sofor_kat['ciddi'].var()
    arac_var = arac_kat['ciddi'].var()
    oran = sof_var / arac_var if arac_var > 0 else 0
    sistem_sonuc.append({
        'kategori': kat, 'ort_ciddi': sub['ciddi_ariza'].mean(),
        'sof_var': sof_var, 'arac_var': arac_var, 'sof_arac_oran': oran
    })
    print(f'{str(kat)[:35]:35s}  {sub["ciddi_ariza"].mean():.3f}      {sof_var:.4f}  {arac_var:.4f}    {oran:.2f}')

sistem_df = pd.DataFrame(sistem_sonuc)

print()
print('=== YORUM ===')
print('Oran > 1.0 = sofor varyansi arac varyansından büyük → sistem soforle daha alakali')
print('Oran < 1.0 = arac dominant')

if len(sistem_df) > 0:
    sof_dominant = sistem_df[sistem_df['sof_arac_oran'] > 1]
    arac_dominant = sistem_df[sistem_df['sof_arac_oran'] < 1]
    print(f'\nSofor-bagli sistem sayisi: {len(sof_dominant)}')
    print(f'Arac-bagli sistem sayisi:  {len(arac_dominant)}')
    if len(sof_dominant) > 0:
        print('\nEn cok sofor-bagli sistemler:')
        print(sof_dominant.nlargest(5, 'sof_arac_oran')[['kategori','sof_arac_oran']].to_string(index=False))


=== SISTEM BAZINDA SOFOR/ARAC VARYANS ANALIZI ===
Her sistem icin: sofor bazinda vs arac bazinda ciddi_oran varyansı

Sistem                               ort_ciddi  sof_var  arac_var  oran(sof/arac)
SOĞUTMA SİSTEMİ ARIZASI              0.268      0.0599  0.0553    1.08
KAPI ARIZALARI                       0.252      0.0584  0.0492    1.19
ELEKTRİK SİSTEMİ ARIZALARI           0.246      0.0605  0.0572    1.06
MOTOR ARIZALARI                      0.548      0.0830  0.0822    1.01
KAROSER ARIZALARI                    0.282      0.0592  0.0608    0.97
KLİMA SİSTEMİ ARIZALARI              0.306      0.0712  0.0664    1.07
FREN ŞİKAYETLERİ                     0.680      0.0787  0.0690    1.14
SÜSPANSİYON SİSTEMİ ARIZALARI        0.479      0.0927  0.0742    1.25
OTOMATİK ŞANZIMAN ARIZALARI          0.555      0.0892  0.0810    1.10
ISITMA SİSTEMİ                       0.291      0.0630  0.0694    0.91
Destek                               1.000      0.0000  0.0000    0.00
AKBİL ARIZALARI    

---

## 8. ML Feature Türetme
11 şoför feature kandidatı: cumulative (her ariza için "o ana kadar geçmiş") + global profil.

In [8]:
# BOLUM 8: ML Feature Turetme — Sofor Bazli
# Her ariza icin "o ana kadar" (gecmis bilgi) sofor feature'lari

df_feat = df.sort_values(['SOFOR_SICILNO', 'OLAYTARIHI']).reset_index(drop=True)

# Cumulative sofor feature'lari
df_feat['sofor_gecmis_n']        = df_feat.groupby('SOFOR_SICILNO').cumcount()
df_feat['sofor_gecmis_ciddi_n']  = df_feat.groupby('SOFOR_SICILNO')['ciddi_ariza'].cumsum().shift(1).fillna(0)
df_feat['sofor_gecmis_ciddi_oran'] = df_feat['sofor_gecmis_ciddi_n'] / df_feat['sofor_gecmis_n'].replace(0, 1)

# Sofor toplam profil (global feature, gelecek görme riski yok — historical leakage'a dikkat)
# Bu feature aslında veri pencermizdeki tum bilgi - ML icin "global sofor riski"
sofor_global = df.groupby('SOFOR_SICILNO').agg(
    sofor_glob_skor=('ciddiyet_skoru', 'mean'),
    sofor_glob_ciddi=('ciddi_ariza', 'mean'),
    sofor_glob_n=('ciddi_ariza', 'count'),
).reset_index()
df_feat = df_feat.merge(sofor_global, on='SOFOR_SICILNO', how='left')

# Sofor-Arac uyumu (cift bazinda gecmis)
df_feat = df_feat.sort_values(['SOFOR_SICILNO', 'KAPINO', 'OLAYTARIHI']).reset_index(drop=True)
df_feat['gecmis_sa_eslesme'] = df_feat.groupby(['SOFOR_SICILNO', 'KAPINO']).cumcount()

# Arac bazinda kac farkli sofor (rotasyon) — global
arac_sofor_n = df.groupby('KAPINO')['SOFOR_SICILNO'].nunique().reset_index()
arac_sofor_n.columns = ['KAPINO', 'arac_basi_sofor_n']
df_feat = df_feat.merge(arac_sofor_n, on='KAPINO', how='left')

# Sofor yogunlasma (global)
df_feat = df_feat.merge(sofor_max[['SOFOR_SICILNO','ana_arac_orani']], on='SOFOR_SICILNO', how='left')

# Sofor toplam farkli arac (kullanim genisligi)
sofor_arac_n = df.groupby('SOFOR_SICILNO')['KAPINO'].nunique().reset_index()
sofor_arac_n.columns = ['SOFOR_SICILNO', 'sofor_farkli_arac']
df_feat = df_feat.merge(sofor_arac_n, on='SOFOR_SICILNO', how='left')

# Yedek sofor flag (50+ arac)
df_feat['yedek_sofor_flag'] = (df_feat['sofor_farkli_arac'] >= 50).astype(int)

# Vardiya
def vardiya_belirle(saat):
    if 6 <= saat < 14: return 0   # Sabah
    elif 14 <= saat < 22: return 1  # Aksam
    else: return 2  # Gece
df_feat['vardiya_kod'] = df_feat['SAAT'].apply(vardiya_belirle)

print('Yeni feature\'lar:')
for f in ['sofor_gecmis_n', 'sofor_gecmis_ciddi_n', 'sofor_gecmis_ciddi_oran',
          'sofor_glob_skor', 'sofor_glob_ciddi', 'gecmis_sa_eslesme',
          'arac_basi_sofor_n', 'ana_arac_orani', 'sofor_farkli_arac',
          'yedek_sofor_flag', 'vardiya_kod']:
    if f in df_feat.columns:
        print(f'  {f}')

print(f'\nFeature seti: {len(df_feat):,} satir')


Yeni feature'lar:
  sofor_gecmis_n
  sofor_gecmis_ciddi_n
  sofor_gecmis_ciddi_oran
  sofor_glob_skor
  sofor_glob_ciddi
  gecmis_sa_eslesme
  arac_basi_sofor_n
  ana_arac_orani
  sofor_farkli_arac
  yedek_sofor_flag
  vardiya_kod

Feature seti: 58,559 satir


---

## 9. Feature Korelasyon Testleri
Her feature × ciddiyet için Pearson + Spearman + Pointbiserial. **ML kararı veriye göre verilir.**

In [9]:
# BOLUM 9: ML Feature Korelasyon Testleri
# Her sofor feature için Pearson + Spearman + Pointbiserial.
# Karar: ekleme/eklememe → veriye gore

features = [
    'sofor_gecmis_n', 'sofor_gecmis_ciddi_n', 'sofor_gecmis_ciddi_oran',
    'sofor_glob_skor', 'sofor_glob_ciddi', 'gecmis_sa_eslesme',
    'arac_basi_sofor_n', 'ana_arac_orani', 'sofor_farkli_arac',
    'yedek_sofor_flag', 'vardiya_kod'
]

print('=== FEATURE × CIDDIYET_SKORU KORELASYON ===')
print(f'{"Feature":28s}  Pearson_r   p        Spearman_rs   p')

sonuclar = []
for f in features:
    if f not in df_feat.columns: continue
    valid = df_feat.dropna(subset=[f])
    r, p = stats.pearsonr(valid[f], valid['ciddiyet_skoru'])
    rs, ps = stats.spearmanr(valid[f], valid['ciddiyet_skoru'])
    rc, pc = stats.pointbiserialr(valid['ciddi_ariza'], valid[f])
    sonuclar.append({
        'feature': f, 'r': r, 'p': p, 'rs': rs, 'ps': ps, 'rc': rc, 'pc': pc,
        'abs_r': abs(r)
    })
    print(f'{f:28s}  {r:+.4f}    {p:.4f}   {rs:+.4f}      {ps:.4f}')

print()
print(f'{"Feature":28s}  CiddiAriza_r  p')
for s in sonuclar:
    print(f'{s["feature"]:28s}  {s["rc"]:+.4f}      {s["pc"]:.4f}')

# Karar verme
print()
print('=== VERI BAZLI KARAR ===')
korr_df = pd.DataFrame(sonuclar).sort_values('abs_r', ascending=False)
print()
print('EKLENEBILIR FEATURE\'LAR (|r| > 0.05 VE p < 0.05):')
guclu = korr_df[(korr_df['abs_r'] > 0.05) & (korr_df['p'] < 0.05)]
if len(guclu) > 0:
    for _, r in guclu.iterrows():
        print(f'  + {r.feature}  (r={r.r:+.4f})')
else:
    print('  Hicbir feature |r|>0.05 esigini gecmiyor.')

print()
print('ZAYIF FEATURE\'LAR (|r| < 0.05 VEYA p >= 0.05):')
zayif = korr_df[(korr_df['abs_r'] <= 0.05) | (korr_df['p'] >= 0.05)]
for _, r in zayif.iterrows():
    print(f'  - {r.feature}  (r={r.r:+.4f}, p={r.p:.4f})')


=== FEATURE × CIDDIYET_SKORU KORELASYON ===
Feature                       Pearson_r   p        Spearman_rs   p
sofor_gecmis_n                +0.0293    0.0000   +0.0299      0.0000
sofor_gecmis_ciddi_n          +0.0366    0.0000   +0.0581      0.0000
sofor_gecmis_ciddi_oran       +0.0071    0.0860   +0.0567      0.0000
sofor_glob_skor               +0.3907    0.0000   +0.3708      0.0000
sofor_glob_ciddi              +0.3242    0.0000   +0.3233      0.0000
gecmis_sa_eslesme             +0.0007    0.8597   +0.0086      0.0373
arac_basi_sofor_n             +0.0297    0.0000   +0.0572      0.0000
ana_arac_orani                -0.0185    0.0000   -0.0408      0.0000
sofor_farkli_arac             +0.0330    0.0000   +0.0324      0.0000
yedek_sofor_flag              +0.0281    0.0000   +0.0207      0.0000
vardiya_kod                   +0.0069    0.0943   -0.0015      0.7079

Feature                       CiddiAriza_r  p
sofor_gecmis_n                +0.0192      0.0000
sofor_gecmis_ciddi_n  

---

## 10. Confounder Kontrolü (Multiple Regression)
Sefer + yaş kontrolü altında şoför feature'ları hâlâ anlamlı mı?

In [10]:
# BOLUM 10: Confounder Kontrolu — Sefer + Yas + Garaj
# Soru: Sofor feature etkisi confounder kontrolu altında hala anlamli mi?
# Yontem: Multiple regression M1→M4

import statsmodels.api as sm
from statsmodels.formula.api import ols

# Sofor seviyesinde aggrege (her sofor icin profil)
sofor_full2 = sofor_full[sofor_full['toplam_ariza'] >= 5].copy()

# Sofor kullanim toplam sefer (ana arac + dagilim)
sofor_full2 = sofor_full2.merge(sofor_max[['SOFOR_SICILNO','ana_arac_orani']], on='SOFOR_SICILNO', how='left')

# Ortalama hatta calistigi araclarin yasi (proxy)
sofor_arac_yas = (
    df.groupby('SOFOR_SICILNO')['MODELYILI']
    .agg(lambda x: x.mode().iloc[0] if len(x) > 0 else 2020)
    .reset_index()
)
sofor_arac_yas['ort_arac_yasi'] = 2025 - sofor_arac_yas['MODELYILI']
sofor_full2 = sofor_full2.merge(sofor_arac_yas[['SOFOR_SICILNO','ort_arac_yasi']], on='SOFOR_SICILNO', how='left')

# Sofor toplam sefer (paylasimli)
sofor_full2['log_sefer'] = np.log1p(sofor_full2['sofor_toplam_arac_sefer'].fillna(0))

regdf = sofor_full2.dropna(subset=['ana_arac_orani','ort_arac_yasi','log_sefer','ort_skor'])
print(f'Regresyon seti: {len(regdf):,} sofor')

# Modeller
m1 = ols('ort_skor ~ ana_arac_orani', data=regdf).fit()
m2 = ols('ort_skor ~ ana_arac_orani + ort_arac_yasi', data=regdf).fit()
m3 = ols('ort_skor ~ ana_arac_orani + ort_arac_yasi + log_sefer', data=regdf).fit()

print()
print(f'{"Model":40s}  ana_arac_k  p_ana   R²')
print(f'{"M1: sadece ana_arac_orani":40s}  {m1.params["ana_arac_orani"]:+.4f}      {m1.pvalues["ana_arac_orani"]:.4f}  {m1.rsquared:.3f}')
print(f'{"M2: + yas":40s}  {m2.params["ana_arac_orani"]:+.4f}      {m2.pvalues["ana_arac_orani"]:.4f}  {m2.rsquared:.3f}')
print(f'{"M3: + yas + sefer":40s}  {m3.params["ana_arac_orani"]:+.4f}      {m3.pvalues["ana_arac_orani"]:.4f}  {m3.rsquared:.3f}')

# Diger feature'lar icin sefer kontrolu
print()
print('=== DIGER FEATURE\'LAR — SEFER KONTROL ALTINDA ===')
for f in ['farkli_arac', 'toplam_ariza']:
    if f not in regdf.columns: continue
    formula = f'ort_skor ~ {f} + log_sefer + ort_arac_yasi'
    m = ols(formula, data=regdf).fit()
    print(f'{f:25s}  k={m.params[f]:+.5f}  p={m.pvalues[f]:.4f}  R²={m.rsquared:.3f}')

print()
print('=== YORUM ===')
p3 = m3.pvalues['ana_arac_orani']
k3 = m3.params['ana_arac_orani']
if p3 < 0.05:
    yon = 'POZITIF' if k3 > 0 else 'NEGATIF'
    print(f'VERIYE GORE: ana_arac_orani sefer ve yas kontrolu altinda hala {yon} ANLAMLI (p={p3:.4f}).')
    if k3 < 0:
        print('  -> Sabit sofor (yuksek ana_arac_orani) = daha az ciddi → sureklilik bonusu')
    else:
        print('  -> Sabit sofor = daha cok ciddi → tek arac suclanmis olabilir')
else:
    print(f'VERIYE GORE: Confounder kontrolu altinda ana_arac_orani etkisi ANLAMSIZ (p={p3:.4f}).')
    print('  -> Yogunlasma etkisi confounder\'dan geliyordu.')


Regresyon seti: 4,615 sofor

Model                                     ana_arac_k  p_ana   R²
M1: sadece ana_arac_orani                 -0.1471      0.0041  0.002
M2: + yas                                 -0.1994      0.0001  0.011
M3: + yas + sefer                         -0.2608      0.0012  0.012

=== DIGER FEATURE'LAR — SEFER KONTROL ALTINDA ===
farkli_arac                k=-0.00062  p=0.7654  R²=0.009
toplam_ariza               k=-0.00231  p=0.1014  R²=0.010

=== YORUM ===
VERIYE GORE: ana_arac_orani sefer ve yas kontrolu altinda hala NEGATIF ANLAMLI (p=0.0012).
  -> Sabit sofor (yuksek ana_arac_orani) = daha az ciddi → sureklilik bonusu


---

## 11. Vaka Analizi — En Riskli vs En İyi Şoförler
Top 10 riskli + Top 10 iyi şoför profil karşılaştırması.

In [11]:
# BOLUM 11: Vaka Analizi — En Riskli ve En Iyi Soforler

# En riskli 10 sofor (ort_skor en yuksek)
filt = sofor_full2[sofor_full2['toplam_ariza'] >= 10]
print(f'Analiz seti: {len(filt):,} sofor (min 10 ariza)')

print()
print('=== EN RISKLI 10 SOFOR (yuksek ort_skor) ===')
top_riskli = filt.nlargest(10, 'ort_skor')[
    ['SOFOR_SICILNO','toplam_ariza','farkli_arac','ana_arac_orani','ort_skor','ciddi_oran']
]
print(top_riskli.to_string(index=False))

print()
print('=== EN IYI 10 SOFOR (dusuk ort_skor, min 10 ariza) ===')
top_iyi = filt.nsmallest(10, 'ort_skor')[
    ['SOFOR_SICILNO','toplam_ariza','farkli_arac','ana_arac_orani','ort_skor','ciddi_oran']
]
print(top_iyi.to_string(index=False))

# Riskli vs Iyi karsilastir
print()
print('=== RISKLI vs IYI SOFOR PROFIL KARSILASTIRMASI ===')
riskli_top = filt.nlargest(50, 'ort_skor')
iyi_top    = filt.nsmallest(50, 'ort_skor')

print(f'{"Metrik":25s}  Riskli (top50)  Iyi (bot50)')
for col in ['toplam_ariza','farkli_arac','ana_arac_orani','ort_skor','ciddi_oran']:
    r_med = riskli_top[col].median()
    i_med = iyi_top[col].median()
    print(f'{col:25s}  {r_med:14.3f}  {i_med:.3f}')

# Mann-Whitney
print()
print('=== MANN-WHITNEY: RISKLI vs IYI ===')
for col in ['farkli_arac','ana_arac_orani']:
    u, p = stats.mannwhitneyu(riskli_top[col].dropna(), iyi_top[col].dropna())
    print(f'{col}: U={u:.0f}, p={p:.4f}')


Analiz seti: 2,588 sofor (min 10 ariza)

=== EN RISKLI 10 SOFOR (yuksek ort_skor) ===
SOFOR_SICILNO  toplam_ariza  farkli_arac  ana_arac_orani  ort_skor  ciddi_oran
      P_56022            12            6        0.583333  5.641667    0.583333
       P_6646            11            7        0.454545  5.420909    0.636364
      P_31869            10           10        0.100000  5.413000    0.900000
      P_50990            12           11        0.166667  5.378333    0.666667
      P_59134            14           14        0.071429  5.304286    0.714286
      P_38069            10            5        0.500000  5.273000    0.600000
       P_9487            16            6        0.687500  5.267500    0.500000
      P_60907            10            5        0.500000  5.264000    0.600000
      P_37977            10            6        0.500000  5.245000    0.800000
      P_37494            18           18        0.055556  5.241111    0.777778

=== EN IYI 10 SOFOR (dusuk ort_skor, min 10 

---

## 12. Time-Based Validation — LEAKAGE TESTİ
`sofor_glob_skor` r=+0.391 çok güçlü. Ama bu **leakage** mi yoksa gerçek prediktif sinyal mi?

**Test:** Veriyi tarih bazında ikiye böl. Train'den (ilk 90 gün) `sofor_glob_skor` hesapla, test'e (sonraki 90 gün) uygula. Korelasyon korunuyor mu?

In [12]:
# BOLUM 12: Time-Based Validation — sofor_glob_skor LEAKAGE Testi
# Soru: sofor_glob_skor (r=+0.391) gercekten prediktif mi yoksa leakage mi?
# Yontem: Time-based split — train ilk yari, test ikinci yari
#         Train'den ogrenilen sofor_glob_skor test'e uygula, korelasyona bak.

# Tarihe gore sirala
df_split = df.sort_values('OLAYTARIHI').reset_index(drop=True)
median_t = df_split['OLAYTARIHI'].median()

train = df_split[df_split['OLAYTARIHI'] < median_t].copy()
test  = df_split[df_split['OLAYTARIHI'] >= median_t].copy()

print('=== TIME-BASED SPLIT ===')
print(f'Train tarih: {train["OLAYTARIHI"].min()} -> {train["OLAYTARIHI"].max()}')
print(f'Test tarih:  {test["OLAYTARIHI"].min()} -> {test["OLAYTARIHI"].max()}')
print(f'Train: {len(train):,} ariza, {train["SOFOR_SICILNO"].nunique():,} sofor')
print(f'Test:  {len(test):,} ariza,  {test["SOFOR_SICILNO"].nunique():,} sofor')

# Coverage: test'te train'de gorulmemis sofor var mi?
train_soforler = set(train['SOFOR_SICILNO'])
test_soforler  = set(test['SOFOR_SICILNO'])
yeni_sofor = test_soforler - train_soforler
ortak = test_soforler & train_soforler
print()
print(f'Test\'te sadece yeni sofor: {len(yeni_sofor):,}')
print(f'Test\'te ortak sofor:      {len(ortak):,}')
yeni_ariza_n = test[test['SOFOR_SICILNO'].isin(yeni_sofor)].shape[0]
print(f'Test\'te yeni soforun ariza sayisi: {yeni_ariza_n:,} ({yeni_ariza_n/len(test)*100:.1f}%)')

# Train'den sofor_glob_skor hesapla
train_sofor_skor = train.groupby('SOFOR_SICILNO').agg(
    train_glob_skor=('ciddiyet_skoru', 'mean'),
    train_glob_ciddi=('ciddi_ariza', 'mean'),
    train_n=('ciddi_ariza', 'count'),
).reset_index()

# Test'e uygula (sadece train'de gorulen soforler)
test_w = test.merge(train_sofor_skor, on='SOFOR_SICILNO', how='left')
test_ortak = test_w[test_w['train_glob_skor'].notna()].copy()
print()
print(f'Test\'te train_skor atanabilen: {len(test_ortak):,} / {len(test):,}')

# Korelasyon (time-aware)
r_aware, p_aware = stats.pearsonr(test_ortak['train_glob_skor'], test_ortak['ciddiyet_skoru'])
rs_aware, ps_aware = stats.spearmanr(test_ortak['train_glob_skor'], test_ortak['ciddiyet_skoru'])
rc_aware, pc_aware = stats.pointbiserialr(test_ortak['ciddi_ariza'], test_ortak['train_glob_skor'])

# Tum-veri korelasyon (leakage'li)
df_w = df.merge(df.groupby('SOFOR_SICILNO').agg(g_skor=('ciddiyet_skoru','mean')).reset_index(), on='SOFOR_SICILNO')
r_full, _ = stats.pearsonr(df_w['g_skor'], df_w['ciddiyet_skoru'])

print()
print('=== LEAKAGE TESTI: TIME-AWARE vs FULL-DATA KORELASYON ===')
print(f'Full-data (leakage\'li):    r={r_full:+.4f}')
print(f'Time-aware (train→test):   r={r_aware:+.4f}  p={p_aware:.6f}')
print(f'Spearman (time-aware):     rs={rs_aware:+.4f}')
print(f'CiddiAriza (time-aware):   r={rc_aware:+.4f}')

dusus = (1 - r_aware/r_full) * 100 if r_full > 0 else 0
print()
print(f'Korelasyon dusus orani: %{dusus:.0f}')

# Yorum
print()
print('=== VERIYE GORE KARAR ===')
if abs(r_aware) > 0.15:
    print(f'-> Time-aware r={r_aware:+.3f} hala GUCLU prediktif.')
    print('   sofor_glob_skor ML icin GUVENILIR feature (target encoding ile).')
elif abs(r_aware) > 0.05:
    print(f'-> Time-aware r={r_aware:+.3f} ZAYIF ama anlamli.')
    print('   ML icin orta degerde feature.')
else:
    print(f'-> Time-aware r={r_aware:+.3f} cok zayıf.')
    print('   sofor_glob_skor BUYUK OLCUDE LEAKAGE\'DAN gelen sinyaldi.')
    print('   ML icin guvenli degil — eklenmemeli veya cok dikkatli kullanilmali.')

# Ek test: aynı sofor train'de cok ariza yaptiysa test'te de aynı pattern var mı?
# Yani sofor profil stability
train_sofor_top = train_sofor_skor[train_sofor_skor['train_n'] >= 5]
test_sofor_skor = test.groupby('SOFOR_SICILNO').agg(
    test_glob_skor=('ciddiyet_skoru','mean'),
    test_n=('ciddi_ariza','count'),
).reset_index()
ortak_df = train_sofor_top.merge(test_sofor_skor, on='SOFOR_SICILNO', how='inner')
ortak_df = ortak_df[ortak_df['test_n'] >= 5]

if len(ortak_df) >= 50:
    r_stab, p_stab = stats.pearsonr(ortak_df['train_glob_skor'], ortak_df['test_glob_skor'])
    print()
    print('=== SOFOR PROFIL STABILITE: train_skor × test_skor ===')
    print(f'n_sofor (her iki donemde min 5 ariza): {len(ortak_df):,}')
    print(f'Pearson r={r_stab:+.4f}, p={p_stab:.6f}')
    if r_stab > 0.5:
        print(f'   STABIL: Sofor profili zamanda korunuyor — gercek pattern.')
    elif r_stab > 0.2:
        print(f'   ORTA STABIL: Pattern var ama gurulu.')
    else:
        print(f'   STABIL DEGIL: Sofor skoru zamana gore degisken — leakage suphesi yuksek.')


=== TIME-BASED SPLIT ===
Train tarih: 2025-01-01 00:30:56 -> 2025-04-07 17:16:17.798000
Test tarih:  2025-04-07 17:19:26.929000 -> 2025-06-30 23:29:56
Train: 29,279 ariza, 5,566 sofor
Test:  29,280 ariza,  5,538 sofor

Test'te sadece yeni sofor: 344
Test'te ortak sofor:      5,194
Test'te yeni soforun ariza sayisi: 1,031 (3.5%)

Test'te train_skor atanabilen: 28,249 / 29,280

=== LEAKAGE TESTI: TIME-AWARE vs FULL-DATA KORELASYON ===
Full-data (leakage'li):    r=+0.3907
Time-aware (train→test):   r=+0.0755  p=0.000000
Spearman (time-aware):     rs=+0.0861
CiddiAriza (time-aware):   r=+0.0484

Korelasyon dusus orani: %81

=== VERIYE GORE KARAR ===
-> Time-aware r=+0.076 ZAYIF ama anlamli.
   ML icin orta degerde feature.

=== SOFOR PROFIL STABILITE: train_skor × test_skor ===
n_sofor (her iki donemde min 5 ariza): 1,745
Pearson r=+0.2241, p=0.000000
   ORTA STABIL: Pattern var ama gurulu.


---

## 13. ASIL HEDEF — Sabit Şoförlü Araç vs Rotasyonlu Araç
Senin asıl sorun: **aynı şoförle sürekli kullanılan araç vs sürekli farklı şoförü olan araç → arıza oranı aynı mı? Hangi sistemler farklı?**

Karşılaştırma:
- **Sabit:** Araçta ≤3 farklı şoför çalışmış
- **Rotasyon:** Araçta ≥20 farklı şoför çalışmış
- Sefer normalize edilmiş arıza oranı (en adil metrik)
- Sistem bazlı arıza dağılımı farkı (Lift)

In [13]:
# BOLUM 13: ASIL HEDEF — Sabit Soforlu Arac vs Cok Rotasyonlu Arac
# Soru: Ayni soforle kullanilan arac vs cok farkli soforu olan arac
# - Ariza orani ayni mi?
# - Hangi sistemlerde farklilasiyor?

# Arac × sofor profili
arac_prof = df.groupby('KAPINO').agg(
    sofor_n=('SOFOR_SICILNO', 'nunique'),
    toplam_ariza=('ciddi_ariza', 'count'),
    ort_skor=('ciddiyet_skoru', 'mean'),
    ciddi_oran=('ciddi_ariza', 'mean'),
).reset_index()

# Sefer ile birleştir (normalize için)
arac_prof = arac_prof.merge(arac_sefer, on='KAPINO', how='left')
arac_prof['ariza_per_1000_sefer'] = arac_prof['toplam_ariza'] / arac_prof['toplam_sefer'] * 1000

# Iki uc grup: sabit (≤3 sofor) vs cok rotasyon (≥20 sofor)
sabit = arac_prof[arac_prof['sofor_n'] <= 3].copy()
rotasyon = arac_prof[arac_prof['sofor_n'] >= 20].copy()
orta = arac_prof[(arac_prof['sofor_n'] > 3) & (arac_prof['sofor_n'] < 20)].copy()

print('=== ASIL HEDEF: SABIT vs ROTASYON ARAC KARSILASTIRMASI ===')
print()
print(f'{"Grup":20s}  n_arac  ort_sofor  ort_ariza  ort_skor  ciddi_oran  ariza/1000sefer')
print(f'{"Sabit (≤3 sofor)":20s}  {len(sabit):6d}  {sabit["sofor_n"].mean():9.1f}  {sabit["toplam_ariza"].mean():9.1f}  {sabit["ort_skor"].mean():.3f}    {sabit["ciddi_oran"].mean():.3f}     {sabit["ariza_per_1000_sefer"].mean():.2f}')
print(f'{"Orta (4-19)":20s}  {len(orta):6d}  {orta["sofor_n"].mean():9.1f}  {orta["toplam_ariza"].mean():9.1f}  {orta["ort_skor"].mean():.3f}    {orta["ciddi_oran"].mean():.3f}     {orta["ariza_per_1000_sefer"].mean():.2f}')
print(f'{"Rotasyon (≥20)":20s}  {len(rotasyon):6d}  {rotasyon["sofor_n"].mean():9.1f}  {rotasyon["toplam_ariza"].mean():9.1f}  {rotasyon["ort_skor"].mean():.3f}    {rotasyon["ciddi_oran"].mean():.3f}     {rotasyon["ariza_per_1000_sefer"].mean():.2f}')

# Mann-Whitney testleri
print()
print('=== ISTATISTIKSEL TESTLER (Sabit vs Rotasyon) ===')

for metrik in ['toplam_ariza', 'ort_skor', 'ciddi_oran', 'ariza_per_1000_sefer']:
    s = sabit[metrik].dropna()
    r = rotasyon[metrik].dropna()
    u, p = stats.mannwhitneyu(s, r, alternative='two-sided')
    yon = 'YUKSEK' if s.median() > r.median() else 'DUSUK'
    fark = s.median() - r.median()
    isaret = '✓' if p < 0.05 else ' '
    print(f'{isaret} {metrik:25s}  Sabit_median={s.median():.3f}  Rotasyon_median={r.median():.3f}  fark={fark:+.3f}  p={p:.4f}  Sabit {yon}')

# SISTEM DETAYI — Hangi sistemler farkli?
print()
print('=== SISTEM BAZLI ARIZA DAGILIMI: Sabit vs Rotasyon ===')

sabit_arac_set = set(sabit['KAPINO'])
rotasyon_arac_set = set(rotasyon['KAPINO'])

sabit_ariza = df[df['KAPINO'].isin(sabit_arac_set)]
rotasyon_ariza = df[df['KAPINO'].isin(rotasyon_arac_set)]

sabit_dist = sabit_ariza['ARIZAUSTKODTANIM'].value_counts(normalize=True) * 100
rotasyon_dist = rotasyon_ariza['ARIZAUSTKODTANIM'].value_counts(normalize=True) * 100

print(f'\n{"Sistem":35s}  Sabit%   Rotasyon%   Fark    Lift')
karsilastirma = []
for kat in sabit_dist.index[:15]:
    s_p = sabit_dist[kat]
    r_p = rotasyon_dist.get(kat, 0)
    fark = s_p - r_p
    lift = s_p / r_p if r_p > 0 else 0
    karsilastirma.append({'kategori': kat, 'sabit': s_p, 'rotasyon': r_p, 'fark': fark, 'lift': lift})
    print(f'{str(kat)[:35]:35s}  {s_p:6.2f}  {r_p:8.2f}    {fark:+.2f}    {lift:.2f}')

karsi_df = pd.DataFrame(karsilastirma)
print()
print('=== Sabit Aracta DAHA COK GORULEN sistemler (Lift > 1.2) ===')
sabit_yuksek = karsi_df[karsi_df['lift'] > 1.2].sort_values('lift', ascending=False)
for _, r in sabit_yuksek.head(5).iterrows():
    print(f'  {r.kategori}: Sabit %{r.sabit:.1f} vs Rotasyon %{r.rotasyon:.1f}  (Lift {r.lift:.2f})')

print()
print('=== Rotasyon Aracta DAHA COK GORULEN sistemler (Lift < 0.8) ===')
rotasyon_yuksek = karsi_df[karsi_df['lift'] < 0.8].sort_values('lift')
for _, r in rotasyon_yuksek.head(5).iterrows():
    print(f'  {r.kategori}: Sabit %{r.sabit:.1f} vs Rotasyon %{r.rotasyon:.1f}  (Lift {r.lift:.2f})')

# Sefer normalize edilmis ariza
print()
print('=== SEFER NORMALIZE ARIZA (en adil karsilastirma) ===')
s_sefer_total = sabit_ariza.merge(arac_sefer, on='KAPINO')['toplam_sefer'].sum() if len(sabit_ariza) > 0 else 0
r_sefer_total = rotasyon_ariza.merge(arac_sefer, on='KAPINO')['toplam_sefer'].sum() if len(rotasyon_ariza) > 0 else 0
print(f'Sabit aracta toplam ariza/1000 sefer:    {len(sabit_ariza) / max(s_sefer_total/1000, 1):.2f}')
print(f'Rotasyon aracta toplam ariza/1000 sefer: {len(rotasyon_ariza) / max(r_sefer_total/1000, 1):.2f}')

# Tutarli sefer dagilimi var mi?
print()
print(f'Sabit araclarin ort. sefer: {sabit["toplam_sefer"].mean():.0f}')
print(f'Rotasyon araclarin ort. sefer: {rotasyon["toplam_sefer"].mean():.0f}')


=== ASIL HEDEF: SABIT vs ROTASYON ARAC KARSILASTIRMASI ===

Grup                  n_arac  ort_sofor  ort_ariza  ort_skor  ciddi_oran  ariza/1000sefer
Sabit (≤3 sofor)         249        2.5        4.8  3.209    0.269     7.85
Orta (4-19)             2307       10.3       13.1  3.553    0.356     10.81
Rotasyon (≥20)           953       26.6       28.4  3.666    0.403     23.86

=== ISTATISTIKSEL TESTLER (Sabit vs Rotasyon) ===
✓ toplam_ariza               Sabit_median=4.000  Rotasyon_median=27.000  fark=-23.000  p=0.0000  Sabit DUSUK
✓ ort_skor                   Sabit_median=3.214  Rotasyon_median=3.654  fark=-0.440  p=0.0000  Sabit DUSUK
✓ ciddi_oran                 Sabit_median=0.250  Rotasyon_median=0.400  fark=-0.150  p=0.0000  Sabit DUSUK
✓ ariza_per_1000_sefer       Sabit_median=2.787  Rotasyon_median=22.324  fark=-19.537  p=0.0000  Sabit DUSUK

=== SISTEM BAZLI ARIZA DAGILIMI: Sabit vs Rotasyon ===

Sistem                               Sabit%   Rotasyon%   Fark    Lift
ELEKTRİK 